# SmartHire — 05. Candidate Fit & Shortlisting Predictor
In this notebook, we address the Shortlisting / Fit Predictor requirement with strict data science integrity:
1. **Scientific Honesty**:
   - Public job datasets contain vacancy postings, but **no proprietary corporate applicant tracking system (ATS) outcome records** (0 = rejected, 1 = shortlisted).
   - We do not fabricate applicant labels.
2. **Explainable Composite Fit Index**:
   - Standard industry practice combining:
     - Textual TF-IDF Cosine Similarity (45%)
     - Deterministic Skill Coverage % (35%)
     - Experience Range Alignment (10%)
     - Domain Role Match (10%)
3. **Calibrated Logistic Regression Prototype**:
   - A benchmark model trained on a calibrated feature distribution to demonstrate how supervised ATS screening functions when historical candidate outcome records are available.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from src.features.match_features import compute_composite_fit_score, parse_experience_range
from src.models.fit_predictor import FitPredictor

fit_pred = FitPredictor.load()
print("FitPredictor engine loaded successfully.")


## 1. Evaluating Candidate Fit Scenarios


In [ ]:
# Scenario A: Strong Candidate (High similarity, high skill coverage, experience match)
strong_fit = fit_pred.evaluate_fit(
    text_similarity=0.78,
    skill_coverage_pct=80.0,
    candidate_exp_years=4.0,
    job_exp_str="3-6 yrs",
    candidate_category="Data Science",
    job_category="Senior Data Scientist"
)
print("--- Scenario A: Strong Fit ---")
print("Composite Score:", strong_fit['composite_score'], "%")
print("Fit Tier:", strong_fit['fit_tier'])
print("Components Breakdown:", strong_fit['components'])
print("Explanation:", strong_fit['explanation'])


In [ ]:
# Scenario B: Developing Candidate (Low similarity, partial skills, experience gap)
developing_fit = fit_pred.evaluate_fit(
    text_similarity=0.35,
    skill_coverage_pct=30.0,
    candidate_exp_years=1.0,
    job_exp_str="4-8 yrs",
    candidate_category="Web Designing",
    job_category="Lead Data Architect"
)
print("--- Scenario B: Developing Fit ---")
print("Composite Score:", developing_fit['composite_score'], "%")
print("Fit Tier:", developing_fit['fit_tier'])
print("Components Breakdown:", developing_fit['components'])
print("Explanation:", developing_fit['explanation'])


## 2. Supervised Benchmark Simulation & Sensitivity Analysis


In [ ]:
# Calibrated benchmark model evaluation
sim_grid = [0.2, 0.4, 0.6, 0.8]
cov_grid = [25.0, 50.0, 75.0, 100.0]

matrix_results = []
for sim in sim_grid:
    row = []
    for cov in cov_grid:
        prob = fit_pred.predict_shortlist_probability(sim, cov)
        row.append(f"{prob * 100:.1f}%")
    matrix_results.append(row)

prob_df = pd.DataFrame(
    matrix_results,
    index=[f"Similarity {s*100:.0f}%" for s in sim_grid],
    columns=[f"Coverage {c:.0f}%" for c in cov_grid]
)
print("Predicted Shortlist Probability Matrix (Simulated Benchmark):")
display(prob_df)


## Conclusion
- The Composite Fit Score provides an interpretable, defensible shortlisting metric directly grounded in candidate qualifications.
- It eliminates the black-box opacity of ungrounded models and provides candidates with clear transparency on how to elevate their market competitiveness.
